# Agent 1 — Tâche 1.2 : charger le modèle de base et mesurer la référence (baseline)

Ce notebook :
1. charge un petit modèle de langage en **4-bit** sur le GPU gratuit de Colab,
2. lui demande de découper des projets en tâches **sans aucun entraînement**,
3. mesure la qualité du format de sa réponse. Ces chiffres seront comparés au modèle fine-tuné (tâche 1.3).

**Avant de commencer** : menu *Exécution → Modifier le type d'exécution → GPU T4*.

In [ ]:
# Cellule 1 — installation (1-2 minutes)
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Cellule 2 — vérifier que le GPU est bien actif
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))
else:
    print("ATTENTION : pas de GPU. Menu Exécution > Modifier le type d'exécution > GPU T4.")

In [ ]:
# Cellule 3 — envoyer tes données (clique sur "Choisir les fichiers")
# Sélectionne val.jsonl (et train.jsonl, test.jsonl) depuis data/processed/ sur ton PC.
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# Cellule 4 — charger le modèle en 4-bit (téléchargement ~6 Go : compte 3 à 6 minutes)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"      # pour comparer : "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # la T4 ne gère pas bien bfloat16
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model.eval()
print(f"Modèle chargé. Mémoire GPU utilisée : {torch.cuda.memory_allocated()/1e9:.2f} Go")

In [ ]:
# Cellule 5 — fonction de génération
def generate(messages, max_new_tokens=1200):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# Cellule 6 — premier essai sur un exemple de validation
import json, random
val = [json.loads(l) for l in open("val.jsonl", encoding="utf-8")]
random.Random(0).shuffle(val)           # mélange pour couvrir plusieurs thèmes

ex = val[0]
print("PROJET :", ex["messages"][1]["content"], "\n")
torch.manual_seed(0)
raw = generate(ex["messages"][:2])      # system + user, sans la réponse attendue
print("RÉPONSE DU MODÈLE BRUT :\n", raw)

In [ ]:
# Cellule 7 — les contrôles de qualité du format
import re

REQUIRED = {"name", "description", "estimate_days", "category"}

def strip_fences(text):
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())

def check_output(raw):
    res = {"json_strict": False, "json_lenient": False, "schema_ok": False,
           "n_tasks": 0, "aberrant_estimates": 0}
    try:
        json.loads(raw); res["json_strict"] = True       # JSON pur, sans rien autour
    except Exception:
        pass
    try:
        data = json.loads(strip_fences(raw)); res["json_lenient"] = True   # JSON après retrait des ```
    except Exception:
        return res
    tasks = data.get("tasks") if isinstance(data, dict) else None
    if isinstance(tasks, list) and tasks and all(isinstance(t, dict) and REQUIRED <= set(t) for t in tasks):
        res["schema_ok"] = True
        res["n_tasks"] = len(tasks)
        res["aberrant_estimates"] = sum(
            1 for t in tasks
            if not isinstance(t["estimate_days"], (int, float)) or not (1 <= t["estimate_days"] <= 15)
        )
    return res

print(check_output(raw))

In [ ]:
# Cellule 8 — mesure de la baseline (compte ~1 minute par exemple sur la T4)
import time
N_EVAL = 8          # monte à 16 pour utiliser tout l'ensemble de validation
torch.manual_seed(0)

rows = []
for ex in val[:N_EVAL]:
    t0 = time.time()
    raw = generate(ex["messages"][:2])
    r = check_output(raw)
    r.update(theme=ex["theme"], seconds=round(time.time() - t0, 1), raw=raw)
    rows.append(r)
    print(f"{ex['theme'][:45]:45s} json={r['json_lenient']!s:5} schema={r['schema_ok']!s:5} taches={r['n_tasks']}")

n = len(rows)
total_tasks = max(sum(r["n_tasks"] for r in rows), 1)
summary = {
    "model": MODEL_ID, "split": "val", "n_eval": n,
    "json_strict_rate": sum(r["json_strict"] for r in rows) / n,
    "json_lenient_rate": sum(r["json_lenient"] for r in rows) / n,
    "schema_ok_rate": sum(r["schema_ok"] for r in rows) / n,
    "aberrant_estimate_rate": sum(r["aberrant_estimates"] for r in rows) / total_tasks,
    "avg_seconds": sum(r["seconds"] for r in rows) / n,
}
print()
for k, v in summary.items():
    print(f"{k:24s} {v:.2f}" if isinstance(v, float) else f"{k:24s} {v}")

In [ ]:
# Cellule 9 — sauvegarder les résultats (à garder dans le projet)
name = "baseline_" + MODEL_ID.split("/")[-1] + ".json"
json.dump({"summary": summary, "rows": rows}, open(name, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
files.download(name)